# 02 — Generate Task Default Params

Exports the `default_params()` of every registered pipeline task to `task_defaults.json`.

**When to run:** whenever a new task class is added to `pipeline_tasks/`.

**Workflow:**
1. Run all cells → `task_defaults.json` is written at the repo root.
2. Copy `task_defaults.json` → `pipeline_config.json` and edit your copy.
3. Use `pipeline_config.json` with `ConfigManager.load()` in the pipeline notebooks.

> The generated file is a **defaults snapshot** — do not edit it directly.

In [1]:
import sys
from pathlib import Path

_repo_root = str(Path("..").resolve())
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import pipeline_tasks
from config_manager import ConfigManager

print("Imports OK")

Imports OK


## Discover tasks

Task classes are read from `pipeline_tasks.__all__`. Adding a new task to
`pipeline_tasks/__init__.py` is all that is needed — this notebook requires no edits.

In [2]:
task_classes = [
    getattr(pipeline_tasks, name)
    for name in pipeline_tasks.__all__
]

print(f"Found {len(task_classes)} task(s):")
for cls in task_classes:
    print(f"  {cls.task_name!r}  deps={cls.dependencies}")

Found 6 task(s):
  'preprocessing'  deps=[]
  'sorting'  deps=['preprocessing']
  'auto_merge'  deps=['sorting']
  'analyzer'  deps=['auto_merge']
  'auto_curation'  deps=['analyzer']
  'burst_detection'  deps=['auto_curation']


## Inspect default params

In [3]:
for cls in task_classes:
    print(f"\n── {cls.task_name} ──")
    for k, v in cls.default_params().items():
        print(f"  {k}: {v!r}")


── preprocessing ──
  output_root: './preprocessed_data'
  bandpass_freq_min: 300
  bandpass_freq_max: 3000
  reference: 'local'
  operator: 'median'
  local_radius: [0, 250]
  dtype: 'float32'
  n_jobs: 30
  chunk_duration: '1s'
  progress_bar: True
  overwrite: True

── sorting ──
  preprocessing_output_root: './preprocessed_data'
  output_root: './spikesorted_data'
  sorter: 'kilosort4'
  docker_image: None
  verbose: True
  remove_existing_folder: True
  delete_output_folder: False
  overwrite: True
  clean_excess_spikes: True
  remove_empty_units: True
  min_high_vram_gb: 14
  high_vram_sorter_kwargs: {'batch_size_seconds': 2.0, 'clear_cache': True, 'invert_sign': True, 'cluster_downsampling': 20, 'max_cluster_subset': None, 'nblocks': 0, 'dmin': 17, 'do_correction': False}
  low_vram_sorter_kwargs: {'batch_size_seconds': 0.5, 'clear_cache': True, 'invert_sign': True, 'cluster_downsampling': 30, 'max_cluster_subset': 50000, 'nblocks': 0, 'do_correction': False}
  sorter_kwargs: {

## Output path

Set `OUTPUT_JSON` to where the defaults snapshot should be written.
The file is always regenerated from scratch.

In [4]:
OUTPUT_JSON = Path("../config/default_tasks_params.json")

## Generate JSON

In [5]:
if OUTPUT_JSON.exists():
    OUTPUT_JSON.unlink()

cm = ConfigManager()
for cls in task_classes:
    cm.register_task(cls)

cm.generate_template(OUTPUT_JSON)
print(f"Written: {OUTPUT_JSON.resolve()}")

Written: /mnt/Vol20tb1/yuxin/codes/Yuxin_MEA/config/default_tasks_params.json


## Verify output

In [6]:
print(OUTPUT_JSON.read_text())

{
  "global": {},
  "tasks": {
    "preprocessing": {
      "output_root": "./preprocessed_data",
      "bandpass_freq_min": 300,
      "bandpass_freq_max": 3000,
      "reference": "local",
      "operator": "median",
      "local_radius": [
        0,
        250
      ],
      "dtype": "float32",
      "n_jobs": 30,
      "chunk_duration": "1s",
      "progress_bar": true,
      "overwrite": true
    },
    "sorting": {
      "preprocessing_output_root": "./preprocessed_data",
      "output_root": "./spikesorted_data",
      "sorter": "kilosort4",
      "docker_image": null,
      "verbose": true,
      "remove_existing_folder": true,
      "delete_output_folder": false,
      "overwrite": true,
      "clean_excess_spikes": true,
      "remove_empty_units": true,
      "min_high_vram_gb": 14,
      "high_vram_sorter_kwargs": {
        "batch_size_seconds": 2.0,
        "clear_cache": true,
        "invert_sign": true,
        "cluster_downsampling": 20,
        "max_cluster_subset":